In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, roc_auc_score
from pgmpy.models import BayesianNetwork
from pgmpy.estimators import HillClimbSearch, BicScore
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination
from skrebate import ReliefF  # Ensure skrebate is installed
from joblib import Parallel, delayed
import warnings
import multiprocessing

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

# Load the dataset
data = pd.read_excel(r"C:\Users\PC\Desktop\PhD\Prospective Study\dataset\class 1\class1_dataset.xlsx")

# Extract predictors (X) and outcome (Y)
X = data.drop('RRI', axis=1)
Y = data['RRI']

# === Step 1: Discretize all features to binary based on global median ===

def discretize_binary_global(X_df):
    """
    Discretize continuous features into binary based on the global median.

    Parameters:
    - X_df: DataFrame containing feature columns.

    Returns:
    - Discretized DataFrame with binary features.
    """
    X_discretized = X_df.copy()
    for column in X_discretized.columns:
        median = X_discretized[column].median()
        X_discretized[column] = (X_discretized[column] > median).astype(int)
    return X_discretized

# Apply global discretization
X_discretized = discretize_binary_global(X)

# Combine discretized features with the outcome
data_discretized = X_discretized.copy()
data_discretized['RRI'] = Y

# Initialize 10-fold cross-validation
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# Define a function to evaluate a single fold for a given k
def evaluate_fold(k, sorted_features, fold, train_index, test_index):
    # Split data into training and testing sets
    train_data = data_discretized.iloc[train_index].copy()
    test_data = data_discretized.iloc[test_index].copy()
    
    # Separate predictors and outcome
    X_train = train_data.drop('RRI', axis=1)
    Y_train = train_data['RRI']
    X_test = test_data.drop('RRI', axis=1)
    Y_test = test_data['RRI']
    
    # Apply Relief on training data
    relief_fold = ReliefF(n_neighbors=100, n_features_to_select='all')
    relief_fold.fit(X_train.values, Y_train.values)
    feature_scores_fold = relief_fold.feature_importances_
    feature_ranking_fold = np.argsort(feature_scores_fold)[::-1]
    sorted_features_fold = X_train.columns[feature_ranking_fold]
    
    # Select top k features based on the current fold's ranking
    selected_features = sorted_features_fold[:k]
    
    # Prepare training and testing data with selected features
    train_selected = train_data[selected_features.tolist() + ['RRI']]
    test_selected = test_data[selected_features.tolist() + ['RRI']]
    
    # Learn the Bayesian Network structure using training data
    hc = HillClimbSearch(train_selected)
    try:
        best_model_structure = hc.estimate(scoring_method=BicScore(train_selected))
    except Exception as e:
        print(f"    HillClimbSearch failed on fold {fold} for k={k}: {e}")
        return None  # Skip this fold if structure learning fails
    
    # Create and fit the Bayesian Network model
    model = BayesianNetwork(best_model_structure.edges())
    try:
        model.fit(train_selected, estimator=BayesianEstimator)
    except Exception as e:
        print(f"    BayesianEstimator failed on fold {fold} for k={k}: {e}")
        return None  # Skip this fold if model fitting fails
    
    # Perform inference
    infer = VariableElimination(model)
    
    # Predict probabilities for the test set
    y_true = test_selected['RRI']
    y_pred_probs = []
    
    network_vars = set(model.nodes())
    
    for _, row in test_selected.iterrows():
        evidence = {col: row[col] for col in selected_features if col in network_vars}
        try:
            result = infer.query(variables=['RRI'], evidence=evidence, show_progress=False)
            # Assuming 'RRI' has states [0, 1]
            y_pred_probs.append(result.values[1])  # Probability of class 1 (positive class)
        except Exception as e:
            print(f"    Error during inference on fold {fold} for k={k}: {e}")
            y_pred_probs.append(0)  # Assign a default probability or handle appropriately
    
    # Convert probabilities to binary predictions
    y_pred = [1 if prob > 0.5 else 0 for prob in y_pred_probs]
    
    # Calculate performance metrics
    try:
        auc = roc_auc_score(y_true, y_pred_probs)
    except ValueError:
        auc = 0.5  # Assign a default AUC if only one class is present
    accuracy = accuracy_score(y_true, y_pred)
    
    return {'auc': auc, 'accuracy': accuracy}

# Define a function to evaluate a single k across all folds
def evaluate_k(k, sorted_features):
    print(f"\nEvaluating top {k} feature(s): {list(sorted_features[:k])}")
    
    # Prepare delayed tasks for each fold
    tasks = []
    for fold, (train_index, test_index) in enumerate(kf.split(data_discretized), 1):
        tasks.append(delayed(evaluate_fold)(k, sorted_features, fold, train_index, test_index))
    
    # Execute folds in parallel
    results = Parallel(n_jobs=-1, verbose=0)(tasks)  # n_jobs=-1 uses all available cores
    
    # Filter out None results due to exceptions
    results = [res for res in results if res is not None]
    
    # Calculate average AUC and accuracy
    if results:
        auc_scores = [res['auc'] for res in results]
        accuracy_scores = [res['accuracy'] for res in results]
        avg_auc = np.mean(auc_scores)
        std_auc = np.std(auc_scores)
        avg_accuracy = np.mean(accuracy_scores)
        std_accuracy = np.std(accuracy_scores)
    else:
        avg_auc = 0
        std_auc = 0
        avg_accuracy = 0
        std_accuracy = 0
    
    print(f"  Average AUC for top {k} feature(s): {avg_auc:.4f} ± {std_auc:.4f}")
    print(f"  Average Accuracy for top {k} feature(s): {avg_accuracy:.4f} ± {std_accuracy:.4f}")
    
    return {'k': k, 'avg_auc': avg_auc, 'std_auc': std_auc, 
            'avg_accuracy': avg_accuracy, 'std_accuracy': std_accuracy}

# Identify all features to enable ranking
num_features = X_discretized.shape[1]
all_features = X_discretized.columns.tolist()

# Initialize a dictionary to store AUC scores for each feature subset size
auc_scores_dict = {}
accuracy_scores_dict = {}

# === Step 2: Feature Ranking using ReliefF on Discretized Features ===

# Initialize Relief for feature ranking
relief = ReliefF(n_neighbors=100, n_features_to_select='all')  # Adjust n_neighbors as needed
relief.fit(X_discretized.values, Y.values)
feature_scores = relief.feature_importances_
feature_ranking = np.argsort(feature_scores)[::-1]  # Indices of features sorted by importance
sorted_features = X_discretized.columns[feature_ranking]

print("Feature ranking based on Relief (overall):")
for rank, feature in enumerate(sorted_features, start=1):
    print(f"{rank}. {feature} (Score: {feature_scores[feature_ranking[rank-1]]:.4f})")

# Define the range of k (number of features to select)
k_values = range(1, num_features + 1)

# === Step 3: Evaluate Different Feature Subset Sizes in Parallel ===

# Set up parallel processing for different k values
# Each k is independent, so they can be processed in parallel
# To avoid excessive memory usage, you might limit the number of parallel jobs
# For example, use n_jobs= multiprocessing.cpu_count() // 2 or similar
# Here, we use all available cores
results_k = Parallel(n_jobs=-1, verbose=10)(
    delayed(evaluate_k)(k, sorted_features) for k in k_values
)

# Process and store the results
for res in results_k:
    k = res['k']
    auc_scores_dict[k] = (res['avg_auc'], res['std_auc'])
    accuracy_scores_dict[k] = (res['avg_accuracy'], res['std_accuracy'])

# Identify the feature subset with the highest average AUC
best_k = max(auc_scores_dict, key=lambda k: auc_scores_dict[k][0])
best_auc, best_auc_std = auc_scores_dict[best_k]
best_features = sorted_features[:best_k]

print("\n=== Feature Selection Results ===")
print(f"Best number of features: {best_k}")
print(f"Best feature subset: {list(best_features)}")
print(f"Best Average AUC: {best_auc:.4f} ± {best_auc_std:.4f}")

Feature ranking based on Relief (overall):
1. rs2252070 (Score: 0.2109)
2. Impact_peak_12 (Score: 0.2027)
3. navicular_drop (Score: 0.1964)
4. EDEQ_total (Score: 0.1945)
5. navicular_drop_asymmetry (Score: 0.1895)
6. rs13946 (Score: 0.1893)
7. Q_angle (Score: 0.1854)
8. average_run_hours (Score: 0.1847)
9. class1_SNP_risk_score (Score: 0.1846)
10. rs9340799 (Score: 0.1816)
11. lower_limb_days_total (Score: 0.1773)
12. Duty_factor_12 (Score: 0.1771)
13. VALR_12 (Score: 0.1729)
14. Q_angle_asymmetry (Score: 0.1666)
15. average_interval_training_frequency (Score: 0.1652)
16. total_ad_ab_ratio (Score: 0.1650)
17. hip_abduction_peak_torque (Score: 0.1545)
18. tracking_period_injury (Score: 0.1539)
19. BMD_spine (Score: 0.1514)
20. rs4789932 (Score: 0.1498)
21. BMI (Score: 0.1431)
22. fat_intake_avg (Score: 0.1408)
23. SC_past_season (Score: 0.1400)
24. rs591058 (Score: 0.1373)
25. sex (Score: 0.1372)
26. Age (Score: 0.1349)
27. knee_extension_peak_torque (Score: 0.1344)
28. knee_flexion_pea

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 tasks      | elapsed: 21.5min
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed: 32.6min
[Parallel(n_jobs=-1)]: Done  17 tasks      | elapsed: 55.3min
[Parallel(n_jobs=-1)]: Done  24 tasks      | elapsed: 75.2min
[Parallel(n_jobs=-1)]: Done  36 out of  39 | elapsed: 550.3min remaining: 45.9min



=== Feature Selection Results ===
Best number of features: 18
Best feature subset: ['rs2252070', 'Impact_peak_12', 'navicular_drop', 'EDEQ_total', 'navicular_drop_asymmetry', 'rs13946', 'Q_angle', 'average_run_hours', 'class1_SNP_risk_score', 'rs9340799', 'lower_limb_days_total', 'Duty_factor_12', 'VALR_12', 'Q_angle_asymmetry', 'average_interval_training_frequency', 'total_ad_ab_ratio', 'hip_abduction_peak_torque', 'tracking_period_injury']
Best Average AUC: 0.6494 ± 0.0389


[Parallel(n_jobs=-1)]: Done  39 out of  39 | elapsed: 815.4min finished
